# Copositive QA-abstention — run on a real LLM (Qwen3-0.6B)

Certified (absolute-threshold) vs calibrated (softmax-confidence) abstention under distribution shift.
**GPU preset. Run all**, paste the printed tables back. (Part 1 ≈10 min; Part 2/LoRA ≈20 min.)


In [ ]:
# 1. deps
!pip -q install -U transformers datasets

In [ ]:
# 2. fetch scripts (public repo, no auth)
!wget -q https://raw.githubusercontent.com/areidyOTH/copositive-colab/main/qa_abstention.py -O qa_abstention.py
print('got qa_abstention.py')

## Part 1 — frozen backbone (R14): train only a read head on the LLM's fixed embeddings


In [ ]:
MODEL = 'Qwen/Qwen3-0.6B'   # fallback if it errors: 'Qwen/Qwen2.5-0.5B'
for s in range(3):
    print(f'\n========== seed {s} ==========')
    !python qa_abstention.py --model {MODEL} --device cuda --pooling last --n_train 4000 --n_eval 2000 --steps 600 --seed {s}

## Part 2 — LoRA-adapt the backbone

Let the **embeddings** adapt (LoRA adapters inside the backbone + head). Trains frozen AND LoRA, both
reads; prints the comparison; **saves all 4 models to `artifacts/` for download.**


In [ ]:
# deps for LoRA. NOTE: Kaggle ships an old torchao that makes peft error on LoRA -> remove it (unused).
!pip -q install -U peft
!pip uninstall -y torchao

In [ ]:
!wget -q https://raw.githubusercontent.com/areidyOTH/copositive-colab/main/qa_abstention.py -O qa_abstention.py
!wget -q https://raw.githubusercontent.com/areidyOTH/copositive-colab/main/qa_abstention_lora.py -O qa_abstention_lora.py
print('got qa_abstention_lora.py')

In [ ]:
# train frozen + LoRA (both reads), comparison table, save artifacts/  (~15-20 min on T4)
!python qa_abstention_lora.py --model {MODEL} --device cuda --pooling last

In [ ]:
# zip the 4 trained models. On Colab this auto-downloads; on Kaggle grab artifacts.zip from the
# right-hand Output/working-dir panel after it finishes.
!zip -qr artifacts.zip artifacts
try:
    from google.colab import files; files.download('artifacts.zip')
except Exception:
    print('artifacts.zip ready in', __import__('os').getcwd(), '-> download from the Output panel')